# 300 m walkability computation

**Goal:** Identify which residential buildings (street number to be precise for each inhabitant in the city) in Bologna are within a **300m walking distance** of a publicly accessible park or green space.

**Pipeline overview:**
1. Load city boundary, buildings street number, parks, buldings polygons and pedestrian network
2. Detect park entrance points (where the pedestrian network intersects park boundaries)
3. Generate 300m walking isochrones from each entrance using GraphHopper
4. Spatial-join buildings against the dissolved isochrones
5. Compute precise building-to-entrance walk distances for matched buildings
6. Export results as GeoJSON, CSV, and SQLite

---

> **GraphHopper must be running before executing the distance cells.**  
> In a separate terminal, navigate to the folder containing `graphhopper-web-11.0.jar`, `config_bologna.yml`, and `Bologna.osm.pbf`, then run:
> ```bash
> rm -rf graph-cache
> java -jar graphhopper-web-11.0.jar server config_bologna.yml
> ```
> Wait for `Started server at HTTP 8989` before continuing.

In [ ]:
!pip install tqdm

  Using cached tqdm-4.67.3-py3-none-any.whl.metadata (57 kB)
Using cached tqdm-4.67.3-py3-none-any.whl (78 kB)
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import pickle
import sqlite3
import time
from concurrent.futures import ThreadPoolExecutor, as_completed

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox
import pandas as pd
import requests
from shapely.geometry import Point, shape
from shapely.strtree import STRtree
from tqdm import tqdm
from shapely.ops import nearest_points

os.chdir("/Users/terezasaskova/Library/Mobile Documents/com~apple~CloudDocs/3-30-300-index")

#Coordinate reference systems 
CRS_WGS    = "EPSG:4326"   # WGS-84 geographic (lat/lon) — required by GraphHopper
CRS_METRIC = "EPSG:32632"  # UTM zone 32N — metric distances in metres

#Analysis thresholds 
WALKING_DISTANCE = 300   # metres — maximum acceptable walk to a park entrance
EUCLID_CUTOFF    = 420   # metres — Euclidean pre-filter; keeps API calls minimal
                         # (straight-line distance is always ≤ walking distance,
                         #  so a ~1.4× buffer safely avoids false negatives)

#GraphHopper endpoints 
ISO_URL = "http://localhost:8989/isochrone"
GH_URL  = "http://localhost:8989/route"

## 1 - Load Input Data

Four data sources are needed:
| Layer | Source | info |
|---|---|---|
| Municipal boundary | `bologna_boundary.gpkg` | area of interest |
| Buildings with civic numbers | `bologna_buildings_streetnum.gpkg` | street numbers |
| Green spaces (three sub-layers) | `bologna_greenery.gpkg` | filtered greenery from DSBN and OSM |
| Buildings polygons | `bologna_buildings.gpkg` | buildings polygons |

+  +open street map - pedestrian paths - walking distance 

In [2]:
#1a. Municipal boundary 
bologna_boundary = gpd.read_file("data/bologna_boundary.gpkg").to_crs(CRS_METRIC)

# WGS-84 version needed by osmnx to download the street network
bologna_wgs84 = bologna_boundary.to_crs(CRS_WGS).geometry.union_all()

In [3]:
# 1b. Civici (address points — one point per street number)
civici = (
    gpd.read_file("data/300m/bologna_buildings_streetnum.gpkg")
    .to_crs(CRS_METRIC)
    .loc[lambda df: df.geometry.notna()]
    .reset_index(drop=True)
)
civici["building_id"] = civici["indirizzo_completo"]
print(f"{len(civici):,} civici loaded")

77,630 civici loaded


In [4]:
#1c. Green spaces
# two layers are merged:
#   pe_unis_all — green spaces - settlment units
#   osm_public — parks from OpenStreetMap - w. private tag - filter out

parks = (
    pd.concat(
        [gpd.read_file("data/300m/bologna_greenery.gpkg", layer=l)
         for l in ["pe_unis_all", "osm_public"]],
        ignore_index=True,
    )
    .to_crs(CRS_METRIC)
    .reset_index(drop=True)
    .assign(park_id=lambda df: df.index)  # unique integer key used throughout
)

print(f"{len(parks):,} parks (before filtering)")
print(parks["layer_type"].value_counts())

1,568 parks (before filtering)
layer_type
osm_public     1026
pe_unis_all     542
Name: count, dtype: int64


## 1. Pedestrian network 
- Download the full walk network for Bologna from OSM and project to UTM 32N.
- The edges GeoDataFrame is used to detect park entrances in the next section.

In [5]:
G = ox.graph_from_polygon(bologna_wgs84, network_type="walk")
G = ox.project_graph(G, to_crs=CRS_METRIC)

_, edges = ox.graph_to_gdfs(G)
edges = edges.reset_index(drop=True)

print(f"{len(edges):,} pedestrian edges")

66,812 pedestrian edges


## 2 Park Entrance 

- available entrances (osm = whenever available) + no explicit entrance (intersection pedestrian crossings x the park boundary)
- fallback for osm parks = centroid * in order to include small greneery within city centre - osm is previously filtered for nonprivate parks/gardens only

Steps:
1. Build an STRtree over all edge geometries 
2. For each park, compute `boundary x edges` 

In [6]:
 
# ── Cell 13 replacement: main entrance loop ────────────────────────────────────
#
# Three-tier fallback, strictly mutually exclusive:
#
#   Tier 1 — Explicit OSM entrance=* nodes ≤ 2 m from the park boundary.
#
#   Tier 2 — Boundary ∩ EXTERNAL pedestrian edges only.
#             "External" = edge whose midpoint lies OUTSIDE the park polygon.
#             This excludes internal park paths (winding roads, alleys inside
#             the park) whose midpoints sit inside the polygon. Those paths do
#             cross the boundary, but using them places the entrance inside the
#             park and GraphHopper routes outward from the wrong side.
#             Nearby duplicate intersection points are merged (≤ 5 m → one).
#
#   Tier 3 — Centroid snapped to nearest EXTERNAL edge.
#             Applied to ALL parks (not only OSM-sourced) as a last resort.
#             Search radius scales with park size so large parks aren't missed.
#
# Note on open/unfenced parks: we do NOT project boundary points onto nearby
# non-crossing edges. Without OSM barrier/access tags we cannot reliably
# distinguish an open park edge from a fenced one, so that heuristic would
# create false entrances through walls. Tier 2 already handles open parks
# correctly when the street edge truly intersects the boundary. When it
# doesn't (street runs just outside), Tier 3 centroid-snap is the safe fallback.
 
from shapely.ops import nearest_points
from shapely.strtree import STRtree
from shapely.geometry import MultiPoint, Point
import numpy as np
from tqdm import tqdm
 
OSM_SNAP_TOL = 2.0   # m — OSM node must be within this of the boundary
DEDUP_DIST   = 5.0   # m — cluster intersection points closer than this
SNAP_RADIUS  = 150   # m — base centroid-snap search radius
 
 
def extract_points(geom) -> list:
    """Extract all Point objects from any Shapely geometry type."""
    if geom is None or geom.is_empty:
        return []
    t = geom.geom_type
    if t == "Point":
        return [geom]
    if t == "MultiPoint":
        return list(geom.geoms)
    if t == "GeometryCollection":
        pts = []
        for g in geom.geoms:
            pts.extend(extract_points(g))
        return pts
    return []
 
 
def cluster_points(pts: list, dist: float) -> list:
    """
    Greedy single-linkage: merge points within `dist` m into one centroid.
    Prevents one path crossing from producing 3–4 near-identical entrance pts.
    """
    if not pts:
        return []
    remaining = list(pts)
    clusters  = []
    while remaining:
        seed  = remaining.pop(0)
        group = [seed]
        far   = []
        for p in remaining:
            (group if seed.distance(p) <= dist else far).append(p)
        remaining = far
        clusters.append(MultiPoint(group).centroid)
    return clusters
 

In [7]:
# ── Cell 11 replacement: fetch OSM entrance nodes ─────────────────────────────
# entrance=True matches the literal string "True" — use explicit string list.
osm_entrances_raw = ox.features_from_polygon(
    bologna_wgs84,
    tags={"entrance": ["yes", "main", "secondary", "service",
                       "staircase", "emergency", "exit"]}
).to_crs(CRS_METRIC)
 
osm_entrance_pts = osm_entrances_raw[
    osm_entrances_raw.geometry.geom_type == "Point"
].copy()
print(f"OSM entrance nodes found: {len(osm_entrance_pts)}")
 

OSM entrance nodes found: 373


## Entrance detection : three fallback steps per park:

1. Explicit OSM entrance=* node within 2 m of park boundary
2. Intersection of park boundary with pedestrian edges
3. Snap park centroid to nearest edge (OSM parks only)

In [56]:
# Pre-compute edge midpoints once (used in every park iteration)
edges_geom = edges.geometry.values
edge_mids  = np.array([
    [g.interpolate(0.5, normalized=True).x,
     g.interpolate(0.5, normalized=True).y]
    for g in edges_geom
])

osm_entrance_tree = STRtree(osm_entrance_pts.geometry.values)
edge_tree         = STRtree(edges_geom)

rows        = []
step_counts = {1: 0, 2: 0, 3: 0, "3_extended": 0}

for _, park in tqdm(parks.iterrows(), total=len(parks), desc="Park entrances"):
    geom     = park.geometry
    boundary = geom.boundary
    park_id  = park["park_id"]
    pts      = []

    # Classify edges: external = midpoint outside the park polygon.
    # Buffer by 1 m so edges that just touch the boundary are included.
    cand_idxs     = edge_tree.query(geom.buffer(1), predicate="intersects")
    external_idxs = [
        i for i in cand_idxs
        if not geom.contains(Point(edge_mids[i, 0], edge_mids[i, 1]))
    ]
    crossing_idxs = [
        i for i in external_idxs
        if boundary.intersects(edges_geom[i])
    ]

    #1:explicit OSM entrance nodes
    osm_hits  = osm_entrance_tree.query(geom.buffer(OSM_SNAP_TOL),
                                        predicate="intersects")
    tier1_pts = [
        osm_entrance_pts.geometry.iloc[i]
        for i in osm_hits
        if boundary.distance(osm_entrance_pts.geometry.iloc[i]) <= OSM_SNAP_TOL
    ]
    if tier1_pts:
        pts = tier1_pts
        step_counts[1] += 1

    #2: boundary ∩ external edges + near-miss projection 
    if not pts:
        tier2_raw = []
        for idx in edge_tree.query(boundary, predicate="intersects"):
            # Skip edges fully contained inside the park (internal park paths)
            if not geom.contains(edges_geom[idx]):
                inter = boundary.intersection(edges_geom[idx])
                tier2_raw.extend(extract_points(inter))

        # Near-miss: edges within 10m but not crossing
        #A park with no fence often has its boundary drawn a few metres inside the actual edge
        #or the street is mapped slightly offset.

        near_miss_idxs = [
            idx for idx in edge_tree.query(boundary.buffer(5), predicate="intersects")
            if not geom.contains(edges_geom[idx])
            and not boundary.intersects(edges_geom[idx])
        ]
        for idx in near_miss_idxs:
            proj_pt, _ = nearest_points(edges_geom[idx], boundary)
            tier2_raw.append(proj_pt)

        if tier2_raw:
            pts = cluster_points(tier2_raw, DEDUP_DIST)
            step_counts[2] += 1

    #3: connect with the nearest external edge (OSM parks only)
    if not pts and park["source"] == "OSM":
        boundary_search = list(edge_tree.query(
            geom.boundary.buffer(SNAP_RADIUS), predicate="intersects"
        ))
        ext_nearby = [
            i for i in boundary_search
            if not geom.contains(Point(edge_mids[i, 0], edge_mids[i, 1]))
        ]

        if ext_nearby:
            closest    = min(ext_nearby,
                             key=lambda i: edges_geom[i].distance(geom.boundary))
            snap_pt, _ = nearest_points(edges_geom[closest], geom.boundary)
            pts.append(snap_pt)
            step_counts[3] += 1

        #Named parks with no walkable edges nearby = expand search radius
        elif pd.notna(park.get("name")) and park.get("name"):
            far_search = list(edge_tree.query(
                geom.boundary.buffer(500), predicate="intersects"
            ))
            ext_far = [
                i for i in far_search
                if not geom.contains(Point(edge_mids[i, 0], edge_mids[i, 1]))
            ]
            if ext_far:
                closest    = min(ext_far,
                                 key=lambda i: edges_geom[i].distance(geom.boundary))
                snap_pt, _ = nearest_points(edges_geom[closest], geom.boundary)
                pts.append(snap_pt)
                step_counts["3_extended"] += 1

    for pt in pts:
        rows.append({"park_id": park_id, "geometry": pt})

entrances = gpd.GeoDataFrame(rows, crs=CRS_METRIC)
missing   = set(parks["park_id"]) - set(entrances["park_id"])

print(f"\n{len(entrances):,} entrance points across "
      f"{entrances['park_id'].nunique()} parks")
print(f"Tiers  →  OSM nodes: {step_counts[1]} | "
      f"boundary∩street: {step_counts[2]} | "
      f"centroid snap: {step_counts[3]} | "
      f"extended snap: {step_counts['3_extended']}")
print(f"Parks with no entrance: {len(missing)}")
if missing:
    print("  Missing IDs:", sorted(missing))

Park entrances: 100%|██████████| 1568/1568 [00:12<00:00, 129.13it/s]


5,224 entrance points across 1409 parks
Tiers  →  OSM nodes: 14 | boundary∩street: 1063 | centroid snap: 331 | extended snap: 1
Parks with no entrance: 159
  Missing IDs: [4, 5, 6, 7, 22, 23, 24, 25, 28, 30, 44, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 60, 61, 63, 66, 67, 69, 74, 78, 88, 89, 93, 94, 98, 99, 101, 116, 117, 120, 127, 128, 129, 131, 133, 134, 137, 138, 139, 140, 141, 142, 145, 154, 155, 157, 158, 159, 160, 161, 163, 164, 174, 184, 186, 187, 199, 228, 230, 232, 233, 234, 235, 237, 238, 239, 240, 241, 257, 267, 285, 286, 296, 297, 299, 302, 309, 312, 319, 322, 334, 335, 378, 379, 385, 386, 388, 395, 402, 403, 408, 409, 410, 411, 412, 413, 414, 415, 416, 417, 418, 419, 422, 423, 424, 425, 426, 427, 428, 429, 430, 440, 441, 445, 446, 457, 458, 459, 461, 462, 464, 468, 476, 480, 482, 483, 484, 487, 489, 490, 491, 499, 505, 506, 510, 515, 523, 525, 526, 527, 528, 529, 530, 531, 532, 533, 540, 638, 1061, 1188]


In [57]:
print(f"Tiers  →  OSM nodes: {step_counts[1]} | "
      f"boundary∩street: {step_counts[2]} | "
      f"centroid snap: {step_counts[3]} | "
      f"extended snap: {step_counts['3_extended']}")

Tiers  →  OSM nodes: 14 | boundary∩street: 1063 | centroid snap: 331 | extended snap: 1


In [58]:

# ── New cell: force-flag civici that physically lie inside a park ──────────────
# These are never caught by the isochrone spatial join (the isochrone expands
# outward from the entrance, not inward). Identify them here; apply the
# override in the final output cell.
 
civici_in_park = gpd.sjoin(
    civici[["building_id", "geometry"]],
    parks[["park_id", "geometry"]],
    how="inner",
    predicate="within",
).drop_duplicates("building_id")
 
civici_in_park_ids    = set(civici_in_park["building_id"])
civici_in_park_lookup = civici_in_park.set_index("building_id")["park_id"].to_dict()
 
print(f"Civici inside a park polygon: {len(civici_in_park_ids):,} "
      f"— will be force-flagged within_300m=True")
 

Civici inside a park polygon: 615 — will be force-flagged within_300m=True


In [43]:
num_parks = set(len(parks))
final = num_parks - missing
print(f"parks: {(num_parks)}")
print(f"filter out: {(missing)}")
print(f"after filtering: {(final)}")


TypeError: 'int' object is not iterable

## 3 - Walking Isochrones (GraphHopper)

For each entrance point, request a **300 m walking isochrone** from the local GraphHopper server. The isochrone is a polygon covering all locations reachable within 300 m of walking from that entrance.



In [59]:
#GraphHopper isochrone
def get_isochrone(lat, lon, distance_m=300):
    """
    Request a distance-based walking isochrone from GraphHopper.
    Returns the GeoJSON polygon feature, or None on failure.
    """
    params = {
        "point":          f"{lat},{lon}",
        "profile":        "foot",
        "distance_limit": distance_m,
        "type":           "json",
    }
    try:
        r = requests.get(ISO_URL, params=params, timeout=10)
        r.raise_for_status()
        return r.json()["polygons"][0]
    except Exception:
        return None


# Convert entrances to WGS84 for GraphHopper
entrances_wgs = entrances.to_crs(CRS_WGS)
entrances["entrance_lat"] = entrances_wgs.geometry.y
entrances["entrance_lon"] = entrances_wgs.geometry.x

iso_rows = []
for _, row in tqdm(entrances.iterrows(), total=len(entrances), desc="Isochrones"):
    result = get_isochrone(row["entrance_lat"], row["entrance_lon"])
    if result:
        iso_rows.append({
            "park_id":      row["park_id"],
            "entrance_lat": row["entrance_lat"],
            "entrance_lon": row["entrance_lon"],
            "geometry":     shape(result["geometry"]),
        })
 
# Isochrones stay in WGS84 from GH, project to metric for spatial join
isochrones = gpd.GeoDataFrame(iso_rows, crs=CRS_WGS).to_crs(CRS_METRIC)
print(f"{len(isochrones):,} isochrones computed "
      f"({len(entrances) - len(isochrones):,} failed)")


Isochrones: 100%|██████████| 5224/5224 [07:39<00:00, 11.37it/s]  


5,224 isochrones computed (0 failed)


## 4 - Spatial Join: Buildings Within 300 m Walk

Each civico that falls inside multiple overlapping isochrones will produce
- multiple rows — one per (civico, entrance) pair.
- this is intentional: Step 5 routes all pairs and keeps the shortest.

In [60]:
civici_gdf = gpd.GeoDataFrame(
    {"building_id": civici["building_id"]},
    geometry=civici.geometry,   # already Point geometries in CRS_METRIC
    crs=CRS_METRIC,
)
 
joined = gpd.sjoin(
    civici_gdf,
    isochrones[["park_id", "entrance_lat", "entrance_lon", "geometry"]],
    how="left",
    predicate="within",
)
 
# All valid (civico, entrance) candidate pairs
candidates = (
    joined[joined["park_id"].notna()]
    .copy()
    .reset_index(drop=True)
)
 
# WGS84 coordinates of each civico for GraphHopper routing
cands_wgs = gpd.GeoDataFrame(
    candidates[["building_id"]],
    geometry=candidates.geometry,
    crs=CRS_METRIC,
).to_crs(CRS_WGS)
candidates["orig_lat"] = cands_wgs.geometry.y
candidates["orig_lon"] = cands_wgs.geometry.x
 
n_pairs = len(candidates)
n_cands = candidates["building_id"].nunique()
print(f"\n{n_cands:,} candidate civici "
      f"({n_cands / len(civici) * 100:.1f}% of total)")
print(f"{n_pairs:,} (civico, entrance) pairs to route")
print(f"avg {n_pairs / max(n_cands, 1):.1f} entrance candidates per civico")
 


61,360 candidate civici (79.0% of total)
576,539 (civico, entrance) pairs to route
avg 9.4 entrance candidates per civico


## 5 - Route ALL (civico, entrance) pairs — keep shortest walk per civico
- GraphHopper gives the true walking distance along the street network.
- Routing to every candidate entrance (not just one) ensures we find the
- genuinely nearest entrance by walking, not by straight-line distance.

In [61]:
#GraphHopper point-to-point routing call 
def graphhopper(orig_lat, orig_lon, dest_lat, dest_lon):
    """
    Return the shortest walking distance (metres) between two lat/lon points.
    Returns None if the request fails or no path is found.
    """
    params = {
        "point":       [f"{orig_lat},{orig_lon}", f"{dest_lat},{dest_lon}"],
        "profile":     "foot",
        "calc_points": "false",   # geometry not needed — distance only
        "instructions": "false",
    }
    try:
        r = requests.get(GH_URL, params=params, timeout=5)
        r.raise_for_status()
        return r.json()["paths"][0]["distance"]
    except Exception:
        return None

In [62]:
results = []
for _, row in tqdm(candidates.iterrows(), total=n_pairs,
                   desc="Routing civico→entrance"):
    dist = graphhopper(
        row["orig_lat"],     row["orig_lon"],
        row["entrance_lat"], row["entrance_lon"],
    )
    if dist is not None:
        results.append({
            "building_id":  row["building_id"],
            "park_id":      row["park_id"],
            "entrance_lat": row["entrance_lat"],
            "entrance_lon": row["entrance_lon"],
            "walk_dist_m":  dist,
        })
 
results_df = pd.DataFrame(results)
print(f"\n{len(results_df):,} pairs routed successfully "
      f"({n_pairs - len(results_df):,} GH failures)")
 
# Per civico: keep the entrance with the shortest WALKING distance
best = (
    results_df
    .sort_values("walk_dist_m")
    .groupby("building_id", sort=False)
    .first()
    .reset_index()
)

Routing civico→entrance: 100%|██████████| 576539/576539 [1:40:17<00:00, 95.82it/s]  



576,539 pairs routed successfully (0 GH failures)


## 6 -  final output 

In [63]:
# ── Cell 23: final output ─────────────────────────────────────────────────────

out = civici.merge(
    best[["building_id", "park_id", "entrance_lat", "entrance_lon", "walk_dist_m"]],
    on="building_id",
    how="left",
)

# which civici were inside at least one isochrone (route was attempted)
out["is_candidate"] = out["building_id"].isin(candidates["building_id"])

out["within_300m"] = out["walk_dist_m"].le(WALKING_DISTANCE)

# Override: civici physically inside a park polygon are within 300 m by definition
in_park_mask = out["building_id"].isin(civici_in_park_ids)
out.loc[in_park_mask, "within_300m"] = True
out.loc[in_park_mask, "walk_dist_m"] = out.loc[in_park_mask, "walk_dist_m"].fillna(0.0)
out.loc[in_park_mask, "park_id"] = (
    out.loc[in_park_mask, "building_id"]
    .map(civici_in_park_lookup)
    .combine_first(out.loc[in_park_mask, "park_id"])
)

# Rename to clean output column names
rename = {
    "ncivsub":            "street_number",
    "indirizzo_completo": "address",
    "quartiere":          "neighbourhood",
    "layer_type":         "park_type",
    "area_ha":            "park_area_ha",
    "source":             "park_source",
    "park_id":            "nearest_park_id",
}
if 'name':
    rename['name'] = "park_name"
out = out.rename(columns=rename)
out = gpd.GeoDataFrame(out, geometry="geometry", crs=CRS_METRIC)

# ── Summary ────────────────────────────────────────────────────────────────────
n_total     = len(out)
n_within    = int(out["within_300m"].sum())
n_in_park   = int(in_park_mask.sum())
n_routed    = int((out["is_candidate"] & out["walk_dist_m"].notna()).sum())
n_no_gh     = int((out["is_candidate"] & out["walk_dist_m"].isna()).sum())
n_no_iso    = int((~out["is_candidate"]).sum())

print(f"Total civici:                    {n_total:,}")
print(f"Within 300 m walk:               {n_within:,}  ({n_within / n_total * 100:.1f}%)")
print(f"  of which routed by GraphHopper:{n_routed:,}")
print(f"  of which inside park (forced): {n_in_park:,}")
print(f"Candidate, no GH route found:    {n_no_gh:,}")
print(f"No isochrone match at all:       {n_no_iso:,}")

Total civici:                    77,630
Within 300 m walk:               59,266  (76.3%)
  of which routed by GraphHopper:61,360
  of which inside park (forced): 615
Candidate, no GH route found:    0
No isochrone match at all:       16,270


## 7 - Export Results
output formats are written to `data/`:

File 

- `buildings_300m_indicator.geojson` 
- `buildings_300m_indicator.csv` 
- `300_walkability_map.gpkg` 

In [64]:
out.to_file("data/300m/300_walkability_map.gpkg",
            driver="GPKG", layer="buildings")
parks[parks["park_id"].isin(out["nearest_park_id"])].to_file(
    "data/300m/300_walkability_map.gpkg", driver="GPKG", layer="parks"
)
 
# GeoJSON + CSV (WGS84)
out_wgs = out.to_crs(CRS_WGS)
out_wgs["longitude"] = out_wgs.geometry.x
out_wgs["latitude"]  = out_wgs.geometry.y
 
final_cols = [
    "street_number", "address", "longitude", "latitude",
    "neighbourhood", "nearest_park_id", "park_type", "park_source",
    "park_area_ha", "entrance_lat", "entrance_lon",
    "walk_dist_m", "within_300m", "geometry",
]
if "park_name" in out_wgs.columns:
    final_cols.insert(final_cols.index("park_type"), "park_name")
 
out_wgs = out_wgs[[c for c in final_cols if c in out_wgs.columns]]
 
out_wgs.to_file("data/buildings_300m_indicator.geojson", driver="GeoJSON")
out_wgs.drop(columns="geometry").to_csv(
    "data/300m/buildings_300m_indicator.csv", index=False
)
 
print("data/300m/300_walkability_map.gpkg  (buildings + parks layers)")
print("data/300m/buildings_300m_indicator.geojson")
print("data/300m/buildings_300m_indicator.csv")
print(out_wgs.head(3).to_string())
 

data/300m/300_walkability_map.gpkg  (buildings + parks layers)
data/300m/buildings_300m_indicator.geojson
data/300m/buildings_300m_indicator.csv
  street_number                             address  longitude   latitude          neighbourhood  nearest_park_id  entrance_lat  entrance_lon  walk_dist_m  within_300m                   geometry
0          27/1        VIA MARCO EMILIO LEPIDO 27/1  11.278541  44.516819  Borgo Panigale - Reno            921.0     44.517364     11.278157       79.001         True  POINT (11.27854 44.51682)
1          46/2  VIA GIOVANNI BATTISTA GANDINO 46/2  11.355349  44.478504          Santo Stefano            216.0     44.479283     11.354519      111.623         True   POINT (11.35535 44.4785)
2           9/2              VIA GUIDO ZUCCHINI 9/2  11.351607  44.501726          Santo Stefano           1397.0     44.501100     11.351751       98.923         True  POINT (11.35161 44.50173)
